# 🧠 Perceptron: รากฐานของโครงข่ายประสาทเทียม

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Perceptron**! ในสมุดบันทึกนี้ เราจะ:
1. กำหนดรูปแบบของโมเดล Perceptron และฟังก์ชันกระตุ้นแบบ Heaviside Step Activation
2. สร้างตัวจำแนกประเภท Perceptron แบบสมบูรณ์ขึ้นมาจากศูนย์
3. ฝึกสอน Perceptron เพื่อเรียนรู้ประตูลอจิก (logical gates): `AND`, `OR` และ `XOR`
4. พล็อตจุดข้อมูลฝึกสอนและ **ขอบเขตการตัดสินใจเชิงเส้น (linear decision boundary)** เพื่อแสดงการแบ่งแยกประเภทที่สำเร็จ
5. สาธิตเหตุผลที่ Perceptron ล้มเหลวในการแก้ปัญหาประตู `XOR` เนื่องจาก **ขีดจำกัดความสามารถในการแยกแยะเชิงเส้น (Linear Separability Limit)**
6. อภิปรายการเปลี่ยนผ่านจากเซลล์ประสาทเดี่ยว (single neurons) ไปสู่ Multi-Layer Perceptrons (MLPs) และโครงข่ายประสาทเชิงลึกสมัยใหม่ (เช่น Fully Connected และ Convolutional Layers ของ YOLO)

มาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อน

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. การเขียนคลาส Perceptron ขึ้นมาจากศูนย์

มาสร้าง Perceptron ที่ติดตามค่าน้ำหนัก (weights), ค่าเอนเอียง (bias) และอัตราการเรียนรู้ (learning rate) และทำการอัปเดตเมื่อเกิดข้อผิดพลาดในการทำนาย

In [ ]:
class ScratchPerceptron:
    def __init__(self, input_dim=2, lr=0.1):
        self.weights = np.zeros(input_dim)
        self.bias = 0.0
        self.lr = lr
        
    def predict(self, x):
        z = np.dot(x, self.weights) + self.bias
        return 1 if z >= 0 else 0
        
    def train(self, X, y, epochs=15):
        history = []
        for epoch in range(epochs):
            errors = 0
            for xi, yi in zip(X, y):
                y_pred = self.predict(xi)
                error = yi - y_pred
                if error != 0:
                    self.weights += self.lr * error * xi
                    self.bias += self.lr * error
                    errors += 1
            history.append(errors)
            if errors == 0:
                break
        return history

## 2. การแก้ปัญหาประตูลอจิก AND และ OR

มาฝึกสอนตัวอย่าง Perceptron สองตัวด้วยข้อมูลประตู `AND` และ `OR` จากนั้นพิมพ์พารามิเตอร์ที่ได้เรียนรู้ออกมา

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

y_and = np.array([0, 0, 0, 1])
y_or = np.array([0, 1, 1, 1])

p_and = ScratchPerceptron()
hist_and = p_and.train(X, y_and, epochs=20)

p_or = ScratchPerceptron()
hist_or = p_or.train(X, y_or, epochs=20)

print("AND Weights:", p_and.weights, "| Bias:", p_and.bias)
print("OR Weights :", p_or.weights, "| Bias:", p_or.bias)

## 3. การแสดงภาพขอบเขตการตัดสินใจ (AND vs. OR vs. XOR)

มาเขียนฟังก์ชันการพล็อตที่ลากเส้นขอบเขตการตัดสินใจ (decision boundary line):
$$w_1 x_1 + w_2 x_2 + b = 0 \implies x_2 = -\frac{w_1}{w_2} x_1 - \frac{b}{w_2}$$

สำหรับประตู `XOR` เราจะแสดงให้เห็นว่าเป็นไปไม่ได้เลยที่จะแยกประเภทข้อมูลกลุ่มนี้ออกจากกันด้วยเส้นตรง

In [ ]:
y_xor = np.array([0, 1, 1, 0])
p_xor = ScratchPerceptron()
p_xor.train(X, y_xor, epochs=50)

def plot_gate_boundary(ax, p, X, y, title):
    for xi, yi in zip(X, y):
        marker = 'o' if yi == 1 else 'x'
        color = 'green' if yi == 1 else 'red'
        ax.scatter(xi[0], xi[1], color=color, marker=marker, s=150, linewidth=3, zorder=5)
        
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('Input 1 (x1)')
    ax.set_ylabel('Input 2 (x2)')
    ax.set_title(title)
    ax.grid(True, linestyle='--', alpha=0.5)
    
    w1, w2 = p.weights[0], p.weights[1]
    b = p.bias
    
    if w2 != 0:
        x1_vals = np.linspace(-0.5, 1.5, 100)
        x2_vals = -(w1 * x1_vals + b) / w2
        ax.plot(x1_vals, x2_vals, color='blue', linewidth=2.5, label='Decision Boundary')
        ax.legend()
    else:
        ax.text(0.5, 0.5, 'Boundary Undefined', color='purple', ha='center', fontsize=12, bbox=dict(facecolor='white', alpha=0.8))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_gate_boundary(axes[0], p_and, X, y_and, "AND Gate (Linearly Separable)")
plot_gate_boundary(axes[1], p_or, X, y_or, "OR Gate (Linearly Separable)")
plot_gate_boundary(axes[2], p_xor, X, y_xor, "XOR Gate (Failed - Non-Separable)")
plt.tight_layout()
plt.show()

ดูที่พล็อตสิ:
-   **ประตู AND และ OR:** เส้นสีน้ำเงินแบ่งข้อมูลออกเป็นสองฝั่งได้อย่างสมบูรณ์แบบ
-   **ประตู XOR:** วงกลมสีเขียว (ค่าเป็น 1) อยู่ที่จุด $(0,1)$ และ $(1,0)$ ขณะที่กากบาทสีแดง (ค่าเป็น 0) อยู่ที่จุด $(0,0)$ และ $(1,1)$ จึงไม่มีเส้นตรงใดๆ ที่จะแบ่งวงกลมสีเขียวไว้ฝั่งหนึ่งและกากบาทสีแดงไว้อีกฝั่งหนึ่งได้เลย!

## 💡 การเชื่อมโยงไปยัง YOLO และการเรียนรู้เชิงลึกสมัยใหม่ (Modern Deep Learning)
*   **ทางออกด้วยโครงข่ายหลายชั้น:** ในการแก้ปัญหา XOR เราจำเป็นต้องซ้อน Perceptron เข้าด้วยกันหลายๆ ชั้น (Multi-Layer Perceptron) พร้อมด้วยฟังก์ชันกระตุ้นแบบไม่เชิงเส้น (เช่น Sigmoid หรือ ReLU)
*   **เลเยอร์สมัยใหม่:** ในโมเดล YOLO ส่วนของ classification และ detection heads จะใช้เลเยอร์ของเซลล์ประสาทที่เชื่อมต่อกัน (Linear layers ตามด้วย SiLU activation) เพื่อสร้างขอบเขตการตัดสินใจที่มีความซับซ้อนและโค้งมนในพื้นที่หลายมิติ ทำให้เครือข่ายสามารถจำแนกประเภทของวัตถุที่ทับซ้อนกันได้